# Classes, Objects & Inheritance

Classes are blueprints, objects are instances created from them, and inheritance lets a child class adopt a parent's properties and methods. The `class` keyword arrived in ES6, but underneath it is still prototypal inheritance — objects delegating to other objects.

Related: [[JS - The new Operator]] · [[JS - Functional Constructors and Errors]]

---

## 1. Classes

A class defines the state (properties) and behaviour (methods) its instances will have.

```js
class Car {
  // The constructor initialises instance properties
  constructor(brand, speed) {
    this.brand = brand;
    this.speed = speed;
  }

  // A method shared by all instances — lives on Car.prototype
  drive() {
    console.log(`${this.brand} is driving at ${this.speed} km/h.`);
  }
}
```

- **`class`** declares it. Also usable as an expression: `const Car = class { ... }`.
- **`constructor`** runs automatically on instantiation. Optional — omit it and you get an implicit empty one.
- **Methods** are written without the `function` keyword and are placed on `Car.prototype`, non-enumerable.

Two things classes do that function constructors don't: the body always runs in **strict mode**, and the class binding is **hoisted but in the TDZ**, so using it before the declaration throws a `ReferenceError` rather than silently working.

### Class fields

```js
class Counter {
  count = 0;              // public field — own property on each instance
  #secret = 'hidden';     // private field — inaccessible outside the class body
  static instances = 0;   // static field — lives on the class itself

  static {                // static initialisation block (ES2022)
    console.log('Counter class evaluated');
  }

  constructor() {
    Counter.instances++;
  }

  get doubled() { return this.count * 2; }   // getter, on the prototype
  set doubled(v) { this.count = v / 2; }

  static reset() { Counter.instances = 0; }  // static method, on the class
}
```

Placement matters:

| Declared as | Lives on | Enumerable |
|---|---|---|
| `this.x = 1` in constructor | the instance | yes |
| `x = 1` class field | the instance | yes |
| `method() {}` | `Class.prototype` | no |
| `get x() {}` | `Class.prototype` | no |
| `static x` / `static m() {}` | the class object | no |

Private fields are enforced by the language, not by convention. Touching `obj.#secret` from outside is a **syntax error**, not a runtime one. To test whether an object has the brand:

```js
static isCounter(obj) { return #secret in obj; }
```

Field initialisers run in source order — public fields before the constructor body in a base class, and immediately after `super()` returns in a derived class.

---

## 2. Objects

An object is a concrete instance built from the blueprint.

```js
const car1 = new Car("Toyota", 120);
const car2 = new Car("Tesla", 150);

car1.brand;    // "Toyota"
car2.drive();  // "Tesla is driving at 150 km/h."
```

`new` allocates the object, links its prototype to `Car.prototype`, and runs the constructor. Each instance carries its own `brand` and `speed`; they *share* the single `drive` function.

```js
car1.hasOwnProperty('brand');  // true
car1.hasOwnProperty('drive');  // false — inherited
car1.drive === car2.drive;     // true — same function object
```

### Losing `this` when a method is detached

This is the most common runtime bug with classes:

```js
const fn = car1.drive;
fn();  // TypeError: Cannot read properties of undefined

setTimeout(car1.drive, 1000);          // same problem
button.addEventListener('click', car1.drive);  // and again
```

`this` is determined by *how* a function is called, not where it's defined. Detaching the method loses the receiver, and since class bodies are strict mode, `this` is `undefined` rather than the global object.

Three fixes:

```js
setTimeout(() => car1.drive(), 1000);        // arrow wrapper — usually best
setTimeout(car1.drive.bind(car1), 1000);     // bind at the call site

class Car {
  drive = () => { ... };  // arrow class field — bound per instance
}
```

The arrow field trades memory for safety: it's an own property, so every instance gets its own copy and it no longer lives on the prototype.

---

## 3. Inheritance

```js
class ElectricVehicle {
  constructor(brand) {
    this.brand = brand;
  }

  charge() {
    console.log(`${this.brand} is charging...`);
  }
}

class ElectricCar extends ElectricVehicle {
  constructor(brand, batteryRange) {
    super(brand);                    // must run before touching 'this'
    this.batteryRange = batteryRange;
  }

  charge() {                         // method overriding
    super.charge();                  // call the parent version first
    console.log(`Battery range will reset to ${this.batteryRange} miles.`);
  }
}

const myEv = new ElectricCar("Rivian", 320);
myEv.charge();
// Rivian is charging...
// Battery range will reset to 320 miles.
```

- **`extends`** establishes the parent-child link.
- **`super(...)`** calls the parent constructor. In a derived class it is mandatory: `this` sits in the TDZ until `super()` returns, so `this.x = 1` before it throws `ReferenceError`.
- **`super.method()`** reaches the parent's version of an overridden method.

### Two things about `super` worth knowing

**Statics are inherited too.** `extends` sets `Object.getPrototypeOf(ElectricCar) === ElectricVehicle`, so static methods flow down:

```js
class Base { static create() { return new this(); } }
class Sub extends Base {}
Sub.create();  // works, and 'this' is Sub — returns a Sub instance
```

**`super` is lexical, not dynamic.** It resolves through the method's `[[HomeObject]]` — the object the method was defined in — not through `this`. So a method ripped out and reattached elsewhere still calls the original parent:

```js
const detached = { charge: ElectricCar.prototype.charge };
detached.charge.call(myEv);  // super.charge still finds ElectricVehicle
```

This is also why `super` only works in shorthand methods, never in a `charge: function () {}` property.

---

## 4. The Prototype Chain

`class` is syntactic sugar. JavaScript never copies properties from a blueprint — it links and delegates.

```
myEv
 └─▶ ElectricCar.prototype        (charge)
      └─▶ ElectricVehicle.prototype   (charge, constructor)
           └─▶ Object.prototype        (toString, hasOwnProperty, ...)
                └─▶ null
```

Looking up `myEv.charge()`:

1. Check `myEv`'s own properties — `brand`, `batteryRange`. Not there.
2. Walk to `ElectricCar.prototype` — found, stop.

If `ElectricCar` hadn't overridden it, the search would continue to `ElectricVehicle.prototype`, then `Object.prototype`, then `null` → `undefined`.

Two consequences:

- **Overriding is shadowing, not replacement.** The parent method still exists; the search just finds the child's first.
- **Changes to a prototype are live.** Adding `ElectricVehicle.prototype.honk = ...` after instances exist makes it immediately available on all of them.

Inspecting the chain:

```js
Object.getPrototypeOf(myEv) === ElectricCar.prototype;             // true
Object.getPrototypeOf(ElectricCar.prototype) === ElectricVehicle.prototype;  // true
myEv instanceof ElectricVehicle;   // true — instanceof walks the whole chain
```

Use `Object.getPrototypeOf()` rather than `__proto__`, which is deprecated legacy.

---

## 5. Beyond the Basics

### Extending built-ins

```js
class Stack extends Array {
  peek() { return this[this.length - 1]; }
}
```

Works natively in ES6+. Methods like `map` and `filter` return a `Stack`, not a plain `Array`, because they consult `Symbol.species`. If you're transpiling down to ES5 this breaks — see the `Reflect.construct` note in [[JS - The new Operator]].

### Mixins — composition where single inheritance falls short

JavaScript allows only one parent. When you need behaviour from several sources, use factory functions that return classes:

```js
const Serializable = (Base) => class extends Base {
  toJSON() { return JSON.stringify(this); }
};
const Comparable = (Base) => class extends Base {
  equals(other) { return this.id === other.id; }
};

class Product extends Serializable(Comparable(Object)) {}
```

### Abstract class guard

```js
class Shape {
  constructor() {
    if (new.target === Shape) throw new TypeError('Shape is abstract');
  }
  area() { throw new Error('area() must be implemented'); }
}
```

### Customising `instanceof`

```js
class Even {
  static [Symbol.hasInstance](n) { return n % 2 === 0; }
}
4 instanceof Even;  // true
```

### A note on when *not* to inherit

Deep inheritance hierarchies are the classic OOP trap — every child is coupled to the parent's implementation, and changing a base class ripples everywhere. In JavaScript, composition (passing collaborators in, or mixing behaviour in as above) is usually the better default. Reach for `extends` when there's a genuine "is-a" relationship, not merely shared code.

---

## References

- [MDN — Using classes](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Using_classes)
- [MDN — Classes reference](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Classes)
- [MDN — Inheritance and the prototype chain](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Inheritance_and_the_prototype_chain)
- [MDN — Private properties](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Classes/Private_properties)
- [MDN — super](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/super)
- [javascript.info — Class inheritance](https://javascript.info/class-inheritance)